# Hybrid choice model

## Model

The model combines a structural latent-variable equation, a choice model, and
measurement equations:

$$
\eta_n=\gamma_1z_{n1}+\gamma_2z_{n2}+\sigma_\eta\nu_n,
$$

$$
U_{nj}=V_{nj}+\delta_j\eta_n+\varepsilon_{nj},\qquad
I_{nq}=a_q+\lambda_q\eta_n+\sigma_q\xi_{nq}.
$$

The likelihood integrates the product of the choice probability and indicator
densities over $\eta_n$. This example uses two structural covariates and two
continuous indicators. One loading and the latent scale are fixed for
identification, while the remaining structural, choice, and measurement
parameters are estimated.

Machine configuration: AMD Ryzen 9 9950X3D CPU (16 cores), 64 GB RAM, and
NVIDIA GeForce RTX 5090 GPU (32 GB VRAM), running Ubuntu 24.04.4 with
PyTorch 2.12.1 and CUDA 13.0. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import pandas as pd
import torch

from IPython.display import HTML, display

from torchdcm import (
    Beta,
    ChoiceDataset,
    ChoiceLatentEffect,
    ContinuousIndicator,
    HybridChoiceModel,
    LatentVariable,
    UtilitySpec,
)
# Use double precision and a fixed seed for stable, reproducible estimation.
torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
# Use CUDA automatically when available; set device = "cpu" to force CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
# Simulate a latent attitude from two structural covariates.
rng = np.random.default_rng(22)
n_obs = 300
z_1 = rng.normal(size=n_obs)
z_2 = rng.normal(size=n_obs)
x_b = rng.normal(size=n_obs)
latent_attitude = 0.55 * z_1 - 0.40 * z_2 + rng.normal(size=n_obs)
# Generate two noisy continuous indicators of the latent attitude.
attitude = latent_attitude + rng.normal(scale=0.60, size=n_obs)
satisfaction = (
    0.30
    + 0.80 * latent_attitude
    + rng.normal(scale=0.70, size=n_obs)
)
# Let the latent attitude enter alternative B utility and sample choices.
utility_b = 0.20 + 0.70 * x_b + 0.60 * latent_attitude
p_b = 1.0 / (1.0 + np.exp(-utility_b))
choice = np.where(rng.uniform(size=n_obs) < p_b, "B", "A")

# Collect choices, structural covariates, and indicators in one frame.
frame = pd.DataFrame(
    {
        "choice": choice,
        "x_a": np.zeros(n_obs),
        "x_b": x_b,
        "z_1": z_1,
        "z_2": z_2,
        "attitude": attitude,
        "satisfaction": satisfaction,
    }
)
data = ChoiceDataset.from_wide(
    frame,
    alternatives=["A", "B"],
    choice="choice",
    variables={"x": {"A": "x_a", "B": "x_b"}},
    obs_variables={
        "z_1": "z_1",
        "z_2": "z_2",
        "attitude": "attitude",
        "satisfaction": "satisfaction",
    },
)


In [3]:
# Fix alternative A utility to set the location of the choice model.
spec = UtilitySpec()
spec.utility("A", Beta("ASC_A", init=0.0, fixed=True))
spec.utility(
    "B",
    Beta("ASC_B", init=0.0)
    + Beta("B_X", init=0.50) * "x",
)

# Specify the structural equation and fix latent scale for identification.
latent_variables = [
    LatentVariable(
        "ATTITUDE",
        intercept=0.0,
        coefficients={
            "z_1": Beta("GAMMA_Z1", init=0.30),
            "z_2": Beta("GAMMA_Z2", init=-0.20),
        },
        sigma_init=1.0,
        sigma_fixed=True,
    )
]
# Link the latent attitude to alternative B utility.
choice_effects = [
    ChoiceLatentEffect(
        "B",
        "ATTITUDE",
        Beta("B_ATTITUDE", init=0.30),
    )
]
# Define measurement equations and fix one loading for orientation.
indicators = [
    ContinuousIndicator(
        "attitude",
        "ATTITUDE",
        intercept=0.0,
        loading=1.0,
        sigma_init=0.60,
        sigma_fixed=False,
    ),
    ContinuousIndicator(
        "satisfaction",
        "ATTITUDE",
        intercept=Beta("SAT_INTERCEPT", init=0.10),
        loading=Beta("SAT_LOADING", init=0.70),
        sigma_init=0.70,
        sigma_fixed=False,
    ),
]

# Integrate choice and indicator likelihoods over latent-variable draws.
model = HybridChoiceModel(
    spec,
    latent_variables=latent_variables,
    choice_effects=choice_effects,
    indicators=indicators,
    n_draws=32,
    seed=22,
    antithetic=True,
    panel=False,
    device=device,
    max_iter=80,
)
result = model.fit(data)
# Render structural, choice, and measurement estimates together.
display(HTML(result.report().to_html()))


Model family,HybridChoiceModel
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,80
Line search,strong_wolfe
